In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# E.3 Universality: Why the Details Didn't Matter

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Epilogue",
    number="E.3",
    title="Universality: Why the Details Didn't Matter",
    blurb="This course spent eight volumes insisting the microscopic details matter — every "
    "coupling, every lattice, every Hamiltonian named to the last term. Here we find the one "
    "regime where they don't. Two Ising models as different as we can make them — square and "
    "triangular lattices, critical temperatures 60% apart — are driven to their transitions "
    "and, after one rescaling, fall onto a single universal curve. We watch a genuine change "
    "to the Hamiltonian move the critical temperature but not the exponents, name the "
    "renormalization group that explains it, and gather the number 1/8 from four computations "
    "spanning classical lattices, an exact solution, and a quantum chain.",
    difficulty="advanced",
    estimate="220–260 min",
)

## Notebook overview

[E.1](oscillator-biography.ipynb) honoured one *object*; [E.2](four-faces-of-the-action.ipynb) found one *principle*; E.3 climbs to the structure of law itself,
and makes the most disorienting claim in the course: the obsessive care about microscopic detail
that eight volumes practised — every coupling named, every lattice specified, every Hamiltonian
written to the last term — was, at the critical point, *unnecessary*. Two microscopically alien
systems land on identical critical exponents, and we demonstrate it rather than assert it.

The vehicle is the 2D Ising model of [§5.10](../05-classical-stat-mech/ising-emergence-universality.ipynb) —
where [§5.10](../05-classical-stat-mech/ising-emergence-universality.ipynb) was after the *meaning* of the transition and left the quantitative apparatus to the
Materials Modelling course, this notebook *builds* that apparatus (finite-size scaling, the Binder
cumulant, the data collapse) and turns it on the deepest question the model answers. We take the
model on **two lattices chosen to be as different as possible**: the **square** lattice (each spin
has 4 neighbours, $T_c = 2.269$) and the **triangular** lattice (6 neighbours, $T_c = 3.641$).
These are not small differences — the critical temperatures differ by 60%, the coordination numbers
by half, the geometry entirely. We sample both by vectorized Metropolis (extending the sampler of
[§5.8](../05-classical-stat-mech/partition-function.ipynb) with the multi-colour update that the
non-bipartite triangular lattice forces), and the raw magnetization curves look **different** —
different $T_c$, different shape. That is the setup for the reveal.

Then we **rescale**: plot $m\,L^{\beta/\nu}$ against $(T - T_c)\,L^{1/\nu}$ with the 2D Ising
exponents $\beta = 1/8$, $\nu = 1$, and the curves for *both* lattices and *all* sizes **collapse**
onto a single universal function — the scaled magnetization at $T_c$ agreeing across sizes to a
couple of percent. The collapse is the point: two systems with different microscopics, different
critical temperatures, and different geometries, described near their transitions by *one* function
of *one* variable — the details did not survive the divergence of the correlation length. We confirm
$T_c$ itself universally by the **Binder cumulant** (a dimensionless crossing, [§5.10](../05-classical-stat-mech/ising-emergence-universality.ipynb)
named it, here it is built), and then make universality's robustness visible: add a next-nearest
diagonal coupling $J_2$ — a genuine change to the Hamiltonian — and watch it **shift $T_c$** while
leaving the exponents *untouched*. Some perturbations are **irrelevant** in the technical sense:
they move the non-universal numbers, not the universal ones. This is the empirical face of the
**renormalization group**, which the notebook names and sketches honestly and does not construct —
the boundary drawn plainly, exactly as [E.2](four-faces-of-the-action.ipynb) named the instanton prefactor as outward.

The finale gathers $\beta = 1/8$ from **four independent computations** spanning the whole course:
square-lattice Monte Carlo, triangular-lattice Monte Carlo, Onsager's exact 1944 solution, and the
order-parameter exponent of the transverse-field Ising *chain* of
[§7.19](../07-quantum-statistical-mechanics/transverse-field-ising.ipynb) — a *quantum* system in
one dimension — connected to the 2D classical model by the quantum–classical mapping of
[§7.20](../07-quantum-statistical-mechanics/imaginary-time-quantum-classical.ipynb). The same number
by four roads, across microscopics *and* across the classical/quantum divide.

> **Conventions (this notebook).** $J = 1$, $k = 1$; the exact critical temperatures are
> $T_c^{\square} = 2/\ln(1+\sqrt2) = 2.2692$ and $T_c^{\triangle} = 4/\ln 3 = 3.6410$; the 2D Ising
> exponents are $\beta = 1/8$, $\nu = 1$, $\gamma = 7/4$. **Sampler.** We use single-flip Metropolis
> (the algorithm of [§5.8](../05-classical-stat-mech/partition-function.ipynb)) with a *vectorized
> multi-colour* sweep: a lattice colouring in which no two same-colour sites are neighbours lets a
> whole colour update at once (checkerboard for the square lattice; **three** colours for the
> triangular, which is *not* bipartite; four for the $J_2$ king-neighbour lattice). Near $T_c$
> Metropolis suffers **critical slowing** — the autocorrelation time grows with $L$ — so we take many
> (fast, vectorized) sweeps, discard a generous burn-in, and **seed-average** the gated quantities to
> tame the noise; a cluster algorithm (Wolff) would decorrelate faster and is named as the standard
> tool, but is not needed at these sizes. **The estimator that matters is the collapse, not any single
> curve:** a finite lattice has only a smooth crossover, and the sharp exponent lives in the data
> collapse across sizes. Neighbour sums are built with `numpy.roll` (the triangular lattice is the
> square lattice *plus one diagonal*; $J_2$ adds *all four* diagonals); the magnetization,
> susceptibility, and Binder cumulant $U = 1 - \langle m^4\rangle/3\langle m^2\rangle^2$ are measured
> with a seeded `numpy.random.default_rng`; the collapse uses each lattice's *own* $T_c$ (the scaling
> *function* is universal, $T_c$ is not); Onsager's closed-form magnetization supplies the exact curve
> (real only below $T_c$ — the branch is guarded); the exponent of
> [§7.19](../07-quantum-statistical-mechanics/transverse-field-ising.ipynb) is *exact* (Pfeuty),
> labelled expected-not-measured.
>
> **How to read the checks.** Each exercise closes with a `validate` call against an independent
> fact: the two exact $T_c$'s; that both lattices are sampled; that the scaled magnetization collapses
> across sizes; that the Binder cumulant is $L$-independent at $T_c$; that $J_2$ shifts $T_c$ but the
> collapse survives with the same $\beta$; and that the four routes agree on $1/8$. A ✓ is strong
> evidence; a ✗ is a prompt to *locate the discrepancy*.
>
> **Scope, and an honest boundary.** We can *demonstrate* universality — measure the collapse, watch
> the irrelevant coupling wash out — but we do **not** construct the renormalization group (no
> Wilsonian momentum-shell integration, no $\varepsilon$-expansion); that machinery is where
> "elementary" ends and the next course begins, named as outward (Wilson; Cardy, *Scaling and
> Renormalization in Statistical Physics*). Small-$L$ Monte Carlo also pins the *exponent value* only
> to finite-size resolution (~15–20%); its precision lives in the collapse and in the exact routes
> (Onsager 1944; Kadanoff for scaling). Cross-reference
> [§5.10](../05-classical-stat-mech/ising-emergence-universality.ipynb) (the Ising source),
> [§5.8](../05-classical-stat-mech/partition-function.ipynb) (Metropolis),
> [§5.9](../05-classical-stat-mech/grand-canonical-ensemble-equivalence.ipynb) (the susceptibility
> fluctuation), [§7.19](../07-quantum-statistical-mechanics/transverse-field-ising.ipynb) (the quantum
> chain), [§7.20](../07-quantum-statistical-mechanics/imaginary-time-quantum-classical.ipynb) (the
> mapping), [§7.21](../07-quantum-statistical-mechanics/path-integral-monte-carlo.ipynb) (blocked error
> bars), and [E.2](four-faces-of-the-action.ipynb) (the honest-boundary discipline); forward to [E.4](how-we-knew.ipynb)
> (the method).

## Theory in brief

### The disorienting claim

Away from a critical point, the microscopic details set everything — the couplings, the lattice, the
interaction range fix the phase, the transition temperature, the response. At a *continuous* phase
transition something else happens: the correlation length $\xi$ diverges, the system fluctuates on
every length scale at once, and under that scale-invariance the microscopic specifics blur away.
What survives is only the coarsest information,

```{math}
:label: eq-e3-universality
\{\text{critical exponents}\} = \mathcal{F}(\text{dimension } d,\ \text{order-parameter symmetry}),
```

so systems that agree on nothing but $d$ and symmetry fall into the same **universality class** and
share *identical* critical exponents. This is the deepest structural fact in the theory of phase
transitions (its explanation, the renormalization group, is named below and constructed in the next
course). The plan of the notebook: demonstrate it, do not assert it.

### Two lattices, chosen to differ

To test the claim we need two systems that are microscopically as unlike as we can arrange yet share
$d = 2$ and $\mathbb{Z}_2$ (up/down) symmetry. The 2D Ising model on the **square** and **triangular**
lattices is ideal: same spins, same symmetry, wildly different geometry, and *exactly known* critical
temperatures,

```{math}
:label: eq-two-lattices
T_c^{\square} = \frac{2}{\ln(1+\sqrt2)} = 2.2692,
\qquad
T_c^{\triangle} = \frac{4}{\ln 3} = 3.6410 ,
```

(the closed forms are Onsager's and Houtappel's exact results). The coordination numbers are 4 and 6,
the $T_c$'s differ by 60% — microscopically nothing alike. We build the triangular lattice as the
square lattice *plus one diagonal* bond direction, which is the cleanest way to see that "6 neighbours"
is "4 neighbours plus a genuine geometric change."

### Finite-size scaling and the collapse

A finite lattice shows no sharp transition — only a smooth crossover, rounded over a width that shrinks
as $L$ grows. The sharp physics is recovered by **finite-size scaling**: near $T_c$ the only length that
matters is the ratio $\xi/L$, so the magnetization of an $L\times L$ system obeys a scaling form whose
content is that the rescaled data,

```{math}
:label: eq-fss
m\,L^{\beta/\nu} = \Phi\!\big((T - T_c)\,L^{1/\nu}\big),
```

is a *single universal function* $\Phi$ of the *single* scaled variable — with the 2D Ising exponents
$\beta = 1/8$, $\nu = 1$. Plot $m\,L^{\beta/\nu}$ against $(T-T_c)\,L^{1/\nu}$ and the curves for every
size, and for *both lattices*, must fall on top of one another. The **collapse is the emblem** of the
notebook: the raw curves differ, the rescaled curves coincide.

### The Binder cumulant

The collapse needs $T_c$, and there is a way to find it that needs no exact solution. The **Binder
cumulant** is a dimensionless ratio of magnetization moments,

```{math}
:label: eq-binder
U = 1 - \frac{\langle m^4\rangle}{3\,\langle m^2\rangle^2},
```

and because it is dimensionless it takes a *universal, $L$-independent* value at the scale-free critical
point. Curves of $U(T)$ for different $L$ therefore **cross** at $T_c$: below it $U \to 2/3$ (ordered),
above it $U \to 0$ (disordered), and only at $T_c$ do all sizes agree. The crossing locates $T_c$ without
knowing it in advance — the standard tool, named in [§5.10](../05-classical-stat-mech/ising-emergence-universality.ipynb)
and built here.

### Relevant and irrelevant

Universality predicts that some changes to the Hamiltonian *do not matter*. Add a next-nearest-neighbour
diagonal coupling $J_2$ — a real microscopic change, more bonds — and the claim is

```{math}
:label: eq-relevant
J_2:\quad T_c \to T_c'(J_2) \ \ (\text{shifts, non-universal}),
\qquad
\{\beta, \nu, \dots\} \ \ (\text{unchanged, universal}) ,
```

the coupling moves the *non-universal* number $T_c$ but leaves the *universal* exponents alone. In the
technical vocabulary $J_2$ is an **irrelevant** perturbation. Temperature, by contrast, is **relevant**:
it drives the system *away* from criticality.

### The renormalization group, named

The explanation is Wilson's. Coarse-grain a system — average spins into blocks, rescale — and its
couplings **flow**; a critical point is a **fixed point** of that flow; the exponents are properties of
the fixed point, so every microscopic model whose flow ends at the same fixed point shares them,

```{math}
:label: eq-rg
\frac{d\{K\}}{d\ell} = \boldsymbol{\beta}(\{K\}),
\qquad
\text{relevant couplings grow, irrelevant ones shrink under } \ell \to \ell + d\ell ,
```

with temperature and field relevant (they grow, taking you off criticality) and the lattice details and
$J_2$ irrelevant (they shrink, wash out). We *demonstrate* the consequences — the collapse, the
irrelevant coupling — but do not build the flow: Wilsonian machinery is the outward boundary, exactly
as [E.2](four-faces-of-the-action.ipynb) named the instanton's fluctuation determinant.

### The four-way β = 1/8 rendezvous

The exponent $\beta = 1/8$ is reached by four roads that share almost nothing,

```{math}
:label: eq-e3-rendezvous
\beta = \tfrac18:\quad
\underbrace{\text{square MC}}_{d=2\ \text{classical}}
\ =\ \underbrace{\text{triangular MC}}_{\text{other lattice}}
\ =\ \underbrace{\text{Onsager 1944}}_{\text{exact}}
\ =\ \underbrace{\text{TFIM chain (§7.19)}}_{d=1\ \text{quantum}} ,
```

the last connected to the first three by the quantum–classical mapping of
[§7.20](../07-quantum-statistical-mechanics/imaginary-time-quantum-classical.ipynb) (a $d$-dimensional
quantum system at $T=0$ maps to a $(d{+}1)$-dimensional classical one — the 1D quantum chain *is* the 2D
classical model in disguise, which is *why* they share the exponent). The same number by two classical
lattices sharing no microscopics, an 80-year-old exact solution, and a quantum chain.

### What E.3 establishes

The course's **unity of behaviour**: at criticality, physics forgets its details, and law has a
structure — the fixed point, the universality class — that sits above any particular Hamiltonian. Three
ascents are then complete: one object, one principle, one structure of law. The Epilogue's last notebook
turns the lens from the physics onto the *method* — not what the course knew, but how it knew it ([E.4](how-we-knew.ipynb)).

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

TC_SQUARE = 2.0 / np.log(1.0 + np.sqrt(2.0))  # Onsager exact, 2.269185
TC_TRI = 4.0 / np.log(3.0)  # Houtappel exact, 3.640957
BETA, NU = 0.125, 1.0  # 2D Ising exponents (beta/nu = 1/8, nu = 1)
SIZES = (12, 18, 24)  # even (square 2-colour) AND divisible by 3 (triangular 3-colour)


def nbr_square(s):
    """Nearest-neighbour spin sum on the square lattice (coordination 4) via numpy.roll."""
    return np.roll(s, 1, 0) + np.roll(s, -1, 0) + np.roll(s, 1, 1) + np.roll(s, -1, 1)


def nbr_one_diagonal(s):
    """The ONE extra diagonal (+1,-1)/(-1,+1) that turns the square lattice triangular (coordination 6)."""
    return np.roll(s, (1, -1), (0, 1)) + np.roll(s, (-1, 1), (0, 1))


def nbr_all_diagonals(s):
    """All FOUR diagonal next-nearest neighbours (the isotropic J2 bonds)."""
    return (
        np.roll(s, (1, -1), (0, 1))
        + np.roll(s, (-1, 1), (0, 1))
        + np.roll(s, (1, 1), (0, 1))
        + np.roll(s, (-1, -1), (0, 1))
    )


def colour_masks(L, kind):
    """Boolean masks of an independent-set colouring: no two same-colour sites are neighbours.

    A whole colour can then be Metropolis-updated simultaneously (exact detailed balance, since
    each site's neighbours are held fixed). Square is bipartite -> 2 colours (checkerboard);
    triangular is NOT bipartite (it has triangles) -> 3 colours via (i+2j) mod 3, valid because
    L is divisible by 3; the J2 king-neighbour lattice needs 4 colours (2x2 blocks).

    Parameters
    ----------
    L : int
        Linear size.
    kind : str
        'square', 'triangular', or 'j2'.

    Returns
    -------
    list of numpy.ndarray
        The colour masks (2, 3, or 4 of them).
    """
    ii, jj = np.indices((L, L))
    if kind == "square":
        c = (ii + jj) % 2
        return [c == 0, c == 1]
    if kind == "triangular":
        c = (ii + 2 * jj) % 3
        return [c == 0, c == 1, c == 2]
    c = (ii % 2) + 2 * (jj % 2)  # 'j2': king neighbours -> 4 colours
    return [c == k for k in range(4)]


def _field(s, kind, J2):
    """The local exchange field Σ_nbr, per lattice kind (J = 1; J2 on the four diagonals)."""
    if kind == "square":
        return nbr_square(s)
    if kind == "triangular":
        return nbr_square(s) + nbr_one_diagonal(s)
    return nbr_square(s) + J2 * nbr_all_diagonals(s)  # 'j2'


def ising_mc(L, T, n_sweeps, burn, rng, kind, J2=0.0):
    """Vectorized multi-colour Metropolis on an L x L Ising lattice; return |m|, <m^2>, <m^4>.

    Extends the single-flip Metropolis of §5.8 with the simultaneous colour update that the
    non-bipartite triangular lattice forces. Each colour is flipped with the Metropolis
    acceptance rng.random < exp(-ΔE/T), ΔE = 2·σ_i·Σ_nbr (J = 1).

    Parameters
    ----------
    L : int
        Linear lattice size (periodic).
    T : float
        Temperature (units J/k).
    n_sweeps, burn : int
        Total sweeps and burn-in discarded before measuring.
    rng : numpy.random.Generator
        Seeded generator (numpy.random.default_rng).
    kind : str
        'square', 'triangular', or 'j2'.
    J2 : float, optional
        Next-nearest diagonal coupling (only used for kind='j2').

    Returns
    -------
    tuple of float
        ``(mean|m|, mean m^2, mean m^4)`` over the post-burn sweeps.
    """
    s = rng.choice(np.array([-1, 1]), size=(L, L))
    beta = 1.0 / T
    masks = colour_masks(L, kind)
    absm, m2, m4 = [], [], []
    for sweep in range(n_sweeps):
        for mask in masks:
            dE = 2.0 * s * _field(s, kind, J2)
            flip = (rng.random(s.shape) < np.exp(-beta * dE)) & mask
            s[flip] *= -1
        if sweep >= burn:
            m = s.mean()
            absm.append(abs(m))
            m2.append(m * m)
            m4.append(m**4)
    return float(np.mean(absm)), float(np.mean(m2)), float(np.mean(m4))


def onsager_magnetization(T):
    """Onsager's exact spontaneous magnetization of the 2D square-lattice Ising model.

    m(T) = (1 - sinh(2/T)^{-4})^{1/8} for T < T_c, else 0 (the branch is guarded: the bracket
    is negative above T_c). The exponent 1/8 is visible in the closed form.
    """
    T = np.atleast_1d(T).astype(float)
    out = np.zeros_like(T)
    below = T < TC_SQUARE
    out[below] = (1.0 - np.sinh(2.0 / T[below]) ** (-4)) ** 0.125
    return out


def binder(m2, m4):
    """The Binder cumulant U = 1 - <m^4>/(3<m^2>^2) (dimensionless; L-independent at T_c)."""
    return 1.0 - m4 / (3.0 * m2**2)


print(
    f"exact critical temperatures:  square {TC_SQUARE:.6f}   triangular {TC_TRI:.6f}   (ratio {TC_TRI/TC_SQUARE:.3f})"
)
print(f"2D Ising exponents: beta = {BETA} (= 1/8), nu = {NU}, beta/nu = {BETA/NU}")

## Exercise 1 — The claim, and two lattices to test it

Away from criticality, details rule; at criticality, maybe not. Cite {eq}`eq-e3-universality`,
{eq}`eq-two-lattices`.

1. State universality (dimension and order-parameter symmetry fix the class; the details wash out at
   the divergent correlation length) and the plan (demonstrate on two deliberately different lattices).
2. Build the square (coordination 4) and triangular (coordination 6 — the square plus one diagonal)
   neighbour rules; state the exact $T_c$'s from {eq}`eq-two-lattices` and confirm them.
3. Draw the two lattices and their $T_c$'s on a shared axis: microscopically, nothing alike.
4. Preview (prose): if universality is real, these two will nonetheless share every critical exponent.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    [TC_SQUARE, TC_TRI],
    [2.0 / np.log(1.0 + np.sqrt(2.0)), 4.0 / np.log(3.0)],
    "two lattices, two exact critical temperatures 60% apart",
    rtol=1e-6,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 2 — Sampling criticality

Measure the order parameter on both lattices. Cite {eq}`eq-fss`.

1. Explain critical slowing (Metropolis autocorrelation grows with $L$ near $T_c$) and the vectorized
   multi-colour sweep that lets us take enough (fast) sweeps; the non-bipartite triangular lattice needs
   three colours (`ising_mc`, reusing the Metropolis of [§5.8](../05-classical-stat-mech/partition-function.ipynb)).
2. Measure $\langle|m|\rangle(T)$ on both lattices across a window bracketing each $T_c$ (`ising_mc`,
   seeded, generous burn-in).
3. Plot the raw $m(T)$ curves for both lattices and several $L$ — visibly *different* ($T_c$ apart,
   shape different).
4. Set up the reveal (prose): these curves share no obvious feature; the next exercise rescales them.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.check(
    curves_sq[24][0] > 0.7 > curves_sq[24][-1]
    and curves_tr[24][0] > 0.7 > curves_tr[24][-1],
    "both lattices sampled: the order parameter falls from ordered to disordered across each T_c",
    f"square {curves_sq[24][0]:.2f}->{curves_sq[24][-1]:.2f}, triangular {curves_tr[24][0]:.2f}->{curves_tr[24][-1]:.2f}",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 3 — The collapse (centerpiece)

One rescaling, and the details vanish. Cite {eq}`eq-fss`.

1. Rescale: plot $m\,L^{\beta/\nu}$ against $(T - T_c)\,L^{1/\nu}$ with $\beta = 1/8$, $\nu = 1$, using
   *each lattice's own* $T_c$ (the subtlety: the scaling function is universal, $T_c$ is not).
2. Show the collapse — both lattices, all $L$, onto one curve — and verify the scaled magnetization at
   $T_c$ agrees across $L$ (seed-averaged for a clean gate).
3. State what the figure *is* (prose): two systems with different microscopics and different $T_c$,
   described near their transitions by one function of one variable.
4. Name the exponents' meaning: $\beta$ (order parameter), $\nu$ (correlation length), and that the
   collapse used both.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    np.concatenate([sc_sq / sc_sq.mean(), sc_tr / sc_tr.mean()]),
    np.ones(2 * len(SIZES)),
    "the collapse: scaled magnetization at T_c is L-independent for both lattices (details washed out)",
    rtol=5e-2,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 4 — Locating $T_c$ without knowing it: the Binder cumulant

A dimensionless crossing. Cite {eq}`eq-binder`.

1. Measure $U = 1 - \langle m^4\rangle/3\langle m^2\rangle^2$ for several $L$ across $T_c$ (`ising_mc`,
   seeded; `binder`).
2. Show the $L$-independent **crossing** at $T_c$: the curves for different sizes meet there, and $U$
   at $T_c$ agrees across $L$ at the universal value ($\approx 0.61$ for 2D Ising).
3. State the tool (prose): $U$ is dimensionless, so at the scale-free critical point it does not depend
   on $L$ — the crossing locates $T_c$ even when it is unknown.
4. Connect to the collapse (one line): the Binder crossing *gives* the $T_c$ the collapse needs.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    U_at_Tc.max() - U_at_Tc.min() < 0.04
    and 0.5 < U_at_Tc.mean() < 0.68
    and np.all(np.diff(U_curves[24]) < 0),
    "the Binder cumulant is L-independent at T_c (a dimensionless crossing) and decreases through it",
    f"U(T_c) across L = {np.round(U_at_Tc, 3)}, spread {U_at_Tc.max() - U_at_Tc.min():.4f}",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 5 — Relevant and irrelevant: what a change to the Hamiltonian does

Move the details, watch the exponents ignore you. Cite {eq}`eq-relevant`, {eq}`eq-rg`.

1. Add a next-nearest diagonal coupling $J_2$ (a genuine microscopic change, all four diagonals) and
   locate its shifted $T_c$ by the Binder crossing (`ising_mc` with `kind='j2'`).
2. Rescale with the *same* exponents and the *new* $T_c$, and show the collapse still holds — the
   exponents are unmoved.
3. State the RG reading (prose, honestly bounded): couplings flow under coarse-graining; $T$ is relevant,
   $J_2$ is irrelevant; critical points are fixed points and the exponents are their properties.
4. Draw the boundary (prose): the course *demonstrates* this; it does not *construct* the RG — Wilson's
   machinery is the next course, exactly as [E.2](four-faces-of-the-action.ipynb) named the instanton's
   fluctuation determinant.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.check(
    Tc_j2 > TC_SQUARE + 0.2 and spread_j2 < 12.0,
    "J2 shifts T_c (relevant: the non-universal number moves) but the collapse survives with the same beta=1/8 (the exponents are irrelevant to it)",
    f"T_c: {TC_SQUARE:.3f} -> {Tc_j2:.3f}; collapse spread at the new T_c = {spread_j2:.1f}%",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 6 — (Student) The four-way rendezvous: one number, four roads

$\beta = 1/8$ from classical lattices, an exact solution, and a quantum chain. Cite {eq}`eq-e3-rendezvous`.

1. Extract $\beta$ from the square- and triangular-lattice finite-size scaling (both $\to 1/8$ within
   finite-$L$ resolution — the honest limit of $L \le 24$).
2. Fit Onsager's exact magnetization near $T_c$ (`onsager_magnetization`) for the exact 2D classical
   value.
3. Bring in the transverse-field Ising *chain* of [§7.19](../07-quantum-statistical-mechanics/transverse-field-ising.ipynb):
   its order-parameter exponent is $1/8$ (Pfeuty, *exact* — labelled expected-not-measured), and the
   mapping of [§7.20](../07-quantum-statistical-mechanics/imaginary-time-quantum-classical.ipynb)
   explains *why* (the 1D quantum chain at $T=0$ *is* the 2D classical Ising model).
4. Assemble the rendezvous and state its meaning (prose): the same $1/8$ across microscopics and across
   the classical/quantum divide — a theorem about what survives scale-invariance, not a coincidence.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    [beta_square, beta_tri],
    [0.125, 0.125],
    "the two Monte-Carlo exponents agree with 1/8 to finite-L (L<=24) resolution",
    rtol=0.25,
)
validate.close(
    beta_onsager, 0.125, "Onsager's exact solution gives beta = 1/8", rtol=1.5e-2
)
validate.check(
    beta_tfim == 0.125,
    "the transverse-field Ising chain (§7.19) gives beta = 1/8 exactly (Pfeuty; via the §7.20 mapping)",
    "a 1D quantum system and a 2D classical one, the same exponent",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 7 — (Synthesis) Unity of behavior

No new computation: what the collapse established.

The course was a long argument that details matter — that to know a system you must write its
Hamiltonian to the last coupling. This notebook is the argument's necessary coda. Away from criticality
the argument holds completely: those couplings set the phase, the transition temperature, the response,
everything. But at a continuous phase transition, where the correlation length swallows every scale, the
details you laboured over become invisible, and systems that agree on nothing but dimension and symmetry
become — for the purposes of their most dramatic behaviour — the same system. We drove two Ising models
with critical temperatures 60% apart onto a single curve; we changed one of their Hamiltonians and
watched the change move a number that was never universal and spare every number that was; and we found
the exponent $1/8$ waiting at the end of four different roads — two lattices, an exact solution, and a
quantum chain that a mapping reveals to have been the same problem all along.

This is **unity of behaviour**: not that the course's systems were secretly identical, but that at the
points where physics is most dramatic it is also most forgetful, and law has a structure — the fixed
point, the universality class — that sits above any particular Hamiltonian. There is a humility in
universality that is easy to miss under the triumph. It says that the thing you can compute exactly — the
coupling, the lattice, the microscopic rule — is, at the one moment everyone cares about, the thing that
does not matter; and the thing that matters — the exponent, the class — is not in your Hamiltonian at
all, but in the geometry of how it flows under rescaling. The course spent its life on the details. Its
deepest lesson is when to let them go.

We also drew the boundary honestly, as this Epilogue has each time. We can *demonstrate* universality —
the collapse is real, the irrelevant coupling washes out — but we did not *construct* the renormalization
group that explains it, and we named that construction as the next course rather than faking it here, the
same instinct that made [E.2](four-faces-of-the-action.ipynb) name the instanton's fluctuation
determinant and every error bar in eight volumes honest.

Three ascents are complete: one object ([E.1](oscillator-biography.ipynb)), one principle
([E.2](four-faces-of-the-action.ipynb)), one structure of law. The Epilogue's last notebook asks the
question underneath all of them — not what the course knew, but *how* it knew it ([E.4](how-we-knew.ipynb)).

## Notebook summary

The Epilogue's third notebook: two Ising models as different as we can make them, one universal curve.

- **The claim** {eq}`eq-e3-universality`: at a continuous transition the correlation length diverges and only
  dimension and order-parameter symmetry survive — the universality class.
- **Two lattices** {eq}`eq-two-lattices`: square (coordination 4, $T_c = 2.269$) and triangular (6,
  $T_c = 3.641$) — a 60% gap, verified exact.
- **The collapse** {eq}`eq-fss`: $m\,L^{\beta/\nu}$ vs $(T-T_c)\,L^{1/\nu}$ with $\beta = 1/8$, $\nu = 1$
  lands both lattices, all sizes, on one universal curve — scaled $m$ at $T_c$ $L$-independent to ~2%
  (gated), across a 60% $T_c$ gap.
- **The Binder cumulant** {eq}`eq-binder`: a dimensionless $L$-independent crossing at $T_c$ (universal
  value $\approx 0.61$, gated) — locating $T_c$ with no exact solution.
- **Relevant and irrelevant** {eq}`eq-relevant`, {eq}`eq-rg`: $J_2$ shifts $T_c$ ($2.269 \to 2.83$,
  gated) but the collapse survives with the same $\beta = 1/8$ — the empirical face of the RG, named and
  honestly not constructed.
- **The four-way rendezvous** {eq}`eq-e3-rendezvous`: $\beta = 1/8$ from square MC, triangular MC, Onsager's
  exact 0.1243, and the TFIM chain of [§7.19](../07-quantum-statistical-mechanics/transverse-field-ising.ipynb)
  (Pfeuty exact, via the [§7.20](../07-quantum-statistical-mechanics/imaginary-time-quantum-classical.ipynb)
  mapping) — across microscopics and across the classical/quantum divide.

What E.3 establishes: the course's **unity of behaviour** — at criticality physics forgets its details,
and law has a structure above any Hamiltonian. The honest edge: universality demonstrated, the RG named
as outward.

## Outlook

- **[E.4](how-we-knew.ipynb) — How We Knew**: the course's epistemology, and the ring's far end (echoing the first line of
  [§0.1](../00-foundations/floating-point.ipynb)).
- **The renormalization group, constructed**: Kadanoff blocking, Wilson's momentum shells, the
  $\varepsilon$-expansion (outward — the next course; Cardy, *Scaling and Renormalization in Statistical
  Physics*).
- **Other universality classes**: XY, Heisenberg, percolation; the roles of symmetry and dimension
  (outward).
- **Universality beyond equilibrium**: dynamical exponents, KPZ growth (outward, named).
- **Cross-reference** [§5.10](../05-classical-stat-mech/ising-emergence-universality.ipynb) (the Ising
  source), [§7.19](../07-quantum-statistical-mechanics/transverse-field-ising.ipynb) (the quantum chain),
  [§7.20](../07-quantum-statistical-mechanics/imaginary-time-quantum-classical.ipynb) (the mapping that
  makes the quantum route agree).

In [ ]:
from ecp.style import footer

footer()